# Pipelines and Cross-Validation

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

np.random.seed(42)

n = 1000

df = pd.DataFrame({
    "customer_id": range(1001, 1001 + n),

    "age": np.random.randint(18, 70, n),

    "monthly_charges": np.round(
        np.random.uniform(20, 150, n), 2
    ),

    "tenure_months": np.random.randint(1, 73, n),

    "support_calls": np.random.poisson(2, n),

    "contract_type": np.random.choice(
        ["Monthly", "Annual", "Two-Year"],
        n,
        p=[0.55, 0.30, 0.15]
    ),

    "internet_service": np.random.choice(
        ["Fiber", "DSL", "None"],
        n,
        p=[0.50, 0.35, 0.15]
    ),

    "payment_method": np.random.choice(
        ["Credit Card", "Bank Transfer", "Electronic Check"],
        n
    )
})

# Generate a realistic-ish churn probability
logit = (
    -1.5
    + 0.025 * df["monthly_charges"]
    - 0.045 * df["tenure_months"]
    + 0.35 * df["support_calls"]
    + 0.9 * (df["contract_type"] == "Monthly")
    + 0.5 * (df["internet_service"] == "Fiber")
)

probability = 1 / (1 + np.exp(-logit))

df["churn"] = np.random.binomial(1, probability)

# Introduce missing values into selected features
for col in ["age", "monthly_charges", "contract_type"]:
    missing_idx = df.sample(frac=0.05, random_state=42).index
    df.loc[missing_idx, col] = np.nan

# Inspect dataset
print(df.shape)
display(df.head())
display(df.isnull().sum())
display(df["churn"].value_counts(normalize=True).round(3))

(1000, 9)


,customer_id,age,monthly_charges,tenure_months,support_calls,contract_type,internet_service,payment_method,churn
0,1001,56.0,89.24,5,5,Annual,DSL,Bank Transfer,1
1,1002,69.0,51.52,65,4,Monthly,None,Bank Transfer,1
2,1003,46.0,55.00,30,1,Annual,DSL,Credit Card,0
3,1004,32.0,69.05,21,0,Monthly,DSL,Electronic Check,1
4,1005,60.0,22.61,70,0,Two-Year,Fiber,Bank Transfer,0


customer_id          0
age                 50
monthly_charges     50
tenure_months        0
support_calls        0
contract_type       50
internet_service     0
payment_method       0
churn                0
dtype: int64

churn
1    0.572
0    0.428
Name: proportion, dtype: float64

In [4]:
# Remove the identifier because it is not a useful predictive feature
X = df.drop(columns=["customer_id", "churn"])
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (800, 7)
Testing features: (200, 7)
Training target: (800,)
Testing target: (200,)


#### Q1. Build a pipeline: Create numerical and categorical preprocessing pipelines, combine them with ColumnTransformer, and add LogisticRegression to a complete Pipeline.

In [7]:
df.columns

Index(['customer_id', 'age', 'monthly_charges', 'tenure_months',
       'support_calls', 'contract_type', 'internet_service', 'payment_method',
       'churn'],
      dtype='object')

In [9]:
df.isnull().sum()

customer_id          0
age                 50
monthly_charges     50
tenure_months        0
support_calls        0
contract_type       50
internet_service     0
payment_method       0
churn                0
dtype: int64

In [11]:
df[df.duplicated()]

,customer_id,age,monthly_charges,tenure_months,support_calls,contract_type,internet_service,payment_method,churn


In [13]:
df[df['customer_id'].duplicated()]

,customer_id,age,monthly_charges,tenure_months,support_calls,contract_type,internet_service,payment_method,churn


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       1000 non-null   int64  
 1   age               950 non-null    float64
 2   monthly_charges   950 non-null    float64
 3   tenure_months     1000 non-null   int32  
 4   support_calls     1000 non-null   int32  
 5   contract_type     950 non-null    object 
 6   internet_service  1000 non-null   object 
 7   payment_method    1000 non-null   object 
 8   churn             1000 non-null   int32  
dtypes: float64(2), int32(3), int64(1), object(3)
memory usage: 58.7+ KB


In [17]:
X.columns

Index(['age', 'monthly_charges', 'tenure_months', 'support_calls',
       'contract_type', 'internet_service', 'payment_method'],
      dtype='object')

In [19]:
num_cols=X_train.select_dtypes(include=['int','float']).columns.tolist()

cat_cols=X_train.select_dtypes(include='object').columns.tolist()
print('Numeric cols: ',num_cols)
print('Categorical cols: ',cat_cols)


Numeric cols:  ['age', 'monthly_charges', 'tenure_months', 'support_calls']
Categorical cols:  ['contract_type', 'internet_service', 'payment_method']


In [27]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [41]:
numeric_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])

categorical_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(handle_unknown='ignore'))
])

In [43]:
preprocessor=ColumnTransformer([
    ('num',numeric_pipeline,num_cols),
    ('cat',categorical_pipeline,cat_cols)
])

In [50]:
from sklearn.linear_model import LogisticRegression
model_pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('classifier',LogisticRegression(max_iter=1000))
])

#### Q2. Fit and predict: Fit the pipeline and display the first 10 predicted churn labels and probabilities.

In [54]:
model_pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'monthly_charges',
                                                   'tenure_months',
                                                   'support_calls']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['contract_type',
                                                   'internet_service',
                                                   'payment_method'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [58]:
y_pred=model_pipeline.predict(X_test)
y_proba=model_pipeline.predict_proba(X_test)

In [70]:
result=pd.DataFrame({
    'Actual Churn':y_test.iloc[:10].to_numpy(),
    'Predicted Churn':y_pred[:10],
    'Churn Probabilty':y_proba[:10,1]
})
result

,Actual Churn,Predicted Churn,Churn Probabilty
0,0,0,0.416164
1,0,0,0.445144
2,1,1,0.735371
3,1,1,0.816444
4,0,0,0.307315
5,0,1,0.514664
6,1,1,0.687205
7,0,0,0.446943
8,0,0,0.165162
9,1,1,0.538209


#### Q3. Evaluate performance: Calculate accuracy, precision, recall, F1-score, and ROC-AUC.

In [73]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score)

In [75]:
print('Accuracy: ',accuracy_score(y_test,y_pred))
print('Precision:',precision_score(y_test,y_pred))
print('Recall:',recall_score(y_test,y_pred))
print('F1 score:',f1_score(y_test,y_pred))
print('ROC-AUC score: ',roc_auc_score(y_test,y_pred))

Accuracy:  0.725
Precision: 0.7657657657657657
Recall: 0.7456140350877193
F1 score: 0.7555555555555555
ROC-AUC score:  0.7216442268461851


Our model achieved an accuracy of 72.5%, meaning it correctly predicted whether customers would churn or remain with the business in 72.5% of test cases. The precision of 76.58% indicates that approximately 76.58% of customers predicted to churn actually belonged to the churn class, while the recall of 74.56% shows that our model successfully identified approximately 74.56% of all actual churners. Our model achieved an F1-score of 75.56%, demonstrating a reasonable balance between precision and recall. Additionally, the ROC-AUC score of 0.7216 suggests that our model has moderate ability to distinguish between customers who churn and those who do not. Overall, our model provides a useful starting point for identifying potential churners, but there is still room for improvement. We can further evaluate our model using a confusion matrix, cross-validation, and classification threshold tuning to improve its performance and support better customer retention decisions.

#### Q5. Five-fold cross-validation: Evaluate the pipeline using cross_val_score with ROC-AUC scoring.

In [89]:
from sklearn.model_selection import cross_val_score

cv_score=cross_val_score(
    model_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring='roc_auc'
)

print("ROC-AUC scores:", cv_score)
print("Mean ROC-AUC:", cv_score.mean())
print("Standard deviation:", cv_score.std())

ROC-AUC scores: [0.80882353 0.84430946 0.79475703 0.77591973 0.76811594]
Mean ROC-AUC: 0.7983851400091809
Standard deviation: 0.02702622966656879


Our model achieved a mean ROC-AUC score of 0.7984 through five-fold cross-validation, indicating good discriminatory ability in distinguishing customers who are likely to churn from those who are likely to remain. The ROC-AUC scores across the five folds ranged from 0.7681 to 0.8443, with a standard deviation of 0.0270, suggesting relatively consistent performance across different data splits. These results indicate that our model generalizes reasonably well across the validation folds, although further evaluation and improvement may help enhance its predictive performance.

#### Q6. Stratified cross-validation: Use StratifiedKFold with five splits, shuffling, and random_state=42

In [92]:
from sklearn.model_selection import StratifiedKFold

In [100]:
skf=StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

stratified_scores=cross_val_score(
    model_pipeline,
    X_train,
    y_train,
    cv=skf,
    scoring='roc_auc'
)
print("Stratified ROC-AUC scores:", stratified_scores)
print("Mean ROC-AUC:", stratified_scores.mean())
print("Standard deviation:", stratified_scores.std())

Stratified ROC-AUC scores: [0.83088235 0.79603581 0.7778133  0.80729416 0.79248288]
Mean ROC-AUC: 0.800901698472031
Standard deviation: 0.01770497529463402


Our model achieved a mean ROC-AUC score of 0.8009 using five-fold stratified cross-validation, indicating good ability to distinguish between customers who are likely to churn and those who are likely to stay. The ROC-AUC scores across the five folds ranged from 0.7778 to 0.8309, while the standard deviation of 0.0177 indicates relatively low variation in performance across the folds. By maintaining approximately the same proportion of churners and non-churners in each fold, stratified cross-validation provides a more representative evaluation, particularly when the target classes are imbalanced. Compared with our previous five-fold cross-validation mean of 0.7984, the stratified approach achieved a slightly higher mean ROC-AUC and lower variability, suggesting marginally more consistent performance across the validation folds.

#### Q7. Compare scores: Compare mean cross-validation ROC-AUC with test-set ROC-AUC.

Our model achieved a mean cross-validation ROC-AUC of 0.8009 using StratifiedKFold, while its test-set ROC-AUC was 0.7216. The cross-validation score is approximately 0.0793 higher than the test-set score. This suggests that our model performed better on the training data's validation folds than on the unseen test set. The difference may indicate some generalization difficulty or differences between the data splits. We should investigate the class distributions and data characteristics, while keeping the test set separate from model tuning.

#### Q8. Inspect transformed features: Retrieve transformed feature names and compare their count with the original feature count.

In [108]:
fitted_preprocessor = model_pipeline.named_steps['preprocessor']

feature_names = fitted_preprocessor.get_feature_names_out()

print("Transformed feature names:")
print(feature_names)

print("\nOriginal feature count:", X_train.shape[1])
print("Transformed feature count:", len(feature_names))

Transformed feature names:
['num__age' 'num__monthly_charges' 'num__tenure_months'
 'num__support_calls' 'cat__contract_type_Annual'
 'cat__contract_type_Monthly' 'cat__contract_type_Two-Year'
 'cat__internet_service_DSL' 'cat__internet_service_Fiber'
 'cat__internet_service_None' 'cat__payment_method_Bank Transfer'
 'cat__payment_method_Credit Card' 'cat__payment_method_Electronic Check']

Original feature count: 7
Transformed feature count: 13


#### Q9. Debug schema mismatches: Check for missing columns, extra columns, and feature-order problems.

In [111]:
# Compare training and testing schemas
train_cols = set(X_train.columns)
test_cols = set(X_test.columns)

missing_in_test = train_cols - test_cols
extra_in_test = test_cols - train_cols

print("Missing columns in X_test:", missing_in_test)
print("Extra columns in X_test:", extra_in_test)

Missing columns in X_test: set()
Extra columns in X_test: set()


#### Q10. Build a reusable evaluator: Write evaluate_pipeline() to return all five classification metrics in a dictionary.